# Capstone build --- Chapter 9: Memory

Each step of the workflow produces something a later step needs: the extraction names the product and issue the policy search queries on, and the classification and policy evidence feed the draft. If the agent recomputed those each time it would parse the message repeatedly and risk two steps disagreeing. Chapter~9 is about the memory that lets a later step read an earlier step's result. In the capstone that memory is the accumulated tool results carried in the agent state.

## The memory abstraction

`WorkingMemory` holds the facts and open questions for a single task, and `ShortTermMemory` is a bounded store that keeps the most recent items. These are the general-purpose structures; a task that accumulates named facts as it runs writes them to a working memory keyed by the task.

In [ ]:
from agentlab.memory.short_term import WorkingMemory, ShortTermMemory
from agentlab.memory.base import MemoryItem, MemoryKind

wm = WorkingMemory(task_id='case-002')
wm.facts['product'] = 'checking_account'
wm.facts['issue'] = 'unauthorized_fee'
print('working facts :', wm.facts)

stm = ShortTermMemory(maxlen=8)
stm.add(MemoryItem(content='classified as complaint', kind=MemoryKind.TOOL))
print('recent items  :', [i.content for i in stm.query('classification', k=3)])

## The capstone's working memory is the state

The `ComplaintAgent` does not reach for a separate store: its working memory is `state.tool_results`, the list of outputs accumulated as the workflow runs. A later step reads an earlier result by index through the agent's `_output` helper. This is what lets step~2 reuse the step~1 extraction instead of parsing the message again.

In [ ]:
from agentlab.capstone.complaint_agent import ComplaintAgent
from agentlab.core.state import AgentState
from agentlab.core.task import TaskSpec

agent = ComplaintAgent()
task = TaskSpec(goal='handle a complaint',
                inputs={'message': 'I was charged a $35 overdraft fee I did not authorize.'})
state = AgentState(task=task, step=2)
# Simulate the memory after classify (step 0) and extract (step 1) have run.
state.tool_results.append({'success': True, 'output': {'category': 'complaint', 'confidence': 1.0}})
state.tool_results.append({'success': True, 'output': {
    'product': 'checking_account', 'issue': 'unauthorized_fee',
    'extraction': {'bound': True}, 'query_facts': [('overdraft_fee', 'has_issue', 'unauthorized_fee')]}})

# At step 2 the agent proposes search_policy, reading the extraction from memory.
action = agent.propose_action(state)
print('proposes     :', action.kind, '->', action.tool_name)
print('reuses step-1 extraction:', action.arguments.get('extraction'))

The extraction computed once at step~1 is carried in the state and read again at step~2, so the message is parsed exactly once and every downstream step grounds on the same facts. The tool results are also what the reasoning trace of Chapter~3 is reconstructed from and what the audit log of Chapter~12 records. Chapter~10 turns the completed sequence of these steps into a trajectory and scores it.